# LeetCode 1403 — Minimum Subsequence in Non-Increasing Order bằng Python

**Link đề:** https://leetcode.com/problems/minimum-subsequence-in-non-increasing-order/
**Độ khó:** Easy · **Chủ đề:** Array, Greedy, Sorting

```python
class Solution:
    def minSubsequence(self, nums: list[int]) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu **subsequence** (dãy con: xóa bớt phần tử, giữ thứ tự) và vì sao ở bài này thứ tự gốc **không quan trọng**.
2. Biến điều kiện "tổng phần chọn > tổng phần còn lại" thành `2 · chosen > total`.
3. Thuật toán **tham lam** (greedy): sắp giảm dần, lấy từ lớn nhất cho tới khi vượt nửa tổng — và **chứng minh** vì sao tham lam đúng.
4. Ba cách cài đặt: `sort`, mảng đếm (counting sort), `heapq`; và cách vét cạn làm "trọng tài" để test.
5. Trace, Time/Space, lỗi hay gặp (`>=` thay `>`, quên sắp xếp output), bài tập.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài

Cho mảng `nums`. Chọn một **dãy con** sao cho tổng các phần tử được chọn **lớn hơn hẳn** (strictly greater) tổng các phần tử **không** được chọn.

Nếu có nhiều đáp án: chọn dãy có **ít phần tử nhất**; nếu vẫn hòa, chọn dãy có **tổng lớn nhất**. Đề bảo đảm đáp án **duy nhất**. Trả về dãy đã **sắp xếp không tăng** (non-increasing = giảm dần, cho phép bằng nhau).

| `nums` | Output | Giải thích |
|---|---|---|
| `[4,3,10,9,8]` | `[10,9]` | tổng 34; `[10,9]=19 > 15`; `[10,8]=18 > 16` cũng 2 phần tử nhưng tổng nhỏ hơn |
| `[4,4,7,6,7]` | `[7,7,6]` | tổng 28; `[7,7]=14` không `> 14`; cần thêm `6` |

**Constraints:** `1 <= nums.length <= 500`, `1 <= nums[i] <= 100`.

In [1]:
examples = [
    ([4, 3, 10, 9, 8], [10, 9]),
    ([4, 4, 7, 6, 7], [7, 7, 6]),
]
for nums, expected in examples:
    print(f"nums = {nums}, tổng = {sum(nums)} -> kỳ vọng {expected}")

nums = [4, 3, 10, 9, 8], tổng = 34 -> kỳ vọng [10, 9]
nums = [4, 4, 7, 6, 7], tổng = 28 -> kỳ vọng [7, 7, 6]


## 2. Nền tảng Python và khái niệm

### 2.1. Subsequence là gì? Vì sao thứ tự ở đây không quan trọng?

**Dãy con** = xóa 0 hoặc nhiều phần tử, phần còn lại **giữ nguyên thứ tự**. Ví dụ `[4,3,10,9,8]` có dãy con `[10, 8]` (xóa 4, 3, 9).

Nhưng đề chỉ quan tâm **tổng** và **số phần tử**, rồi bắt trả về dạng **đã sắp giảm dần**. Vậy việc chọn dãy con thực chất là chọn một **tập con các vị trí** — thứ tự gốc không ảnh hưởng đáp án.

In [2]:
nums = [4, 3, 10, 9, 8]
chosen_positions = [2, 4]                     # chọn nums[2]=10 và nums[4]=8
sub = [nums[i] for i in chosen_positions]     # list comprehension
rest = [nums[i] for i in range(len(nums)) if i not in chosen_positions]
print("dãy con:", sub, "tổng", sum(sub))
print("phần còn lại:", rest, "tổng", sum(rest))

dãy con: [10, 8] tổng 18
phần còn lại: [4, 3, 9] tổng 16


`[nums[i] for i in chosen_positions]` là **list comprehension**: tạo list mới bằng cách lấy `nums[i]` cho từng `i`.

### 2.2. Biến đổi điều kiện: `chosen > rest` ⇔ `2 · chosen > total`

Vì `rest = total − chosen`, nên `chosen > total − chosen` ⇔ `2 · chosen > total`. Ta **không cần** tính `rest` riêng, chỉ cần `total` một lần.

Dùng `2 * chosen > total` thay vì `chosen > total / 2` để tránh số thực.

In [3]:
total = sum([4, 4, 7, 6, 7])
for chosen in [14, 15, 20]:
    print(f"chosen={chosen}: rest={total - chosen}, chosen > rest? {chosen > total - chosen}, 2*chosen > total? {2 * chosen > total}")

chosen=14: rest=14, chosen > rest? False, 2*chosen > total? False
chosen=15: rest=13, chosen > rest? True, 2*chosen > total? True
chosen=20: rest=8, chosen > rest? True, 2*chosen > total? True


### 2.3. Sắp xếp giảm dần: `sorted(nums, reverse=True)` và `list.sort(reverse=True)`

- `sorted(x, reverse=True)` → list **mới**, giữ nguyên `x`.
- `x.sort(reverse=True)` → sắp **tại chỗ**, trả `None`.

In [4]:
nums = [4, 3, 10, 9, 8]
print("sorted:", sorted(nums, reverse=True), "| nums vẫn:", nums)
nums.sort(reverse=True)
print("sau nums.sort:", nums)
print("giá trị trả về của .sort():", [1, 3, 2].sort())

sorted: [10, 9, 8, 4, 3] | nums vẫn: [4, 3, 10, 9, 8]
sau nums.sort: [10, 9, 8, 4, 3]
giá trị trả về của .sort(): None


### 2.4. `heapq` — hàng đợi ưu tiên (heap) trong Python

`heapq` biến một list thành **min-heap**: `heappop` luôn lấy ra phần tử **nhỏ nhất** trong `O(log n)`. Muốn lấy **lớn nhất** thì lưu số **âm** (`-x`): phần tử nhỏ nhất của `-x` chính là `x` lớn nhất.

`heapify(list)` dựng heap trong `O(n)`.

In [5]:
import heapq

h = [-x for x in [4, 3, 10, 9, 8]]
heapq.heapify(h)
print("heap (số âm):", h)
print("lấy lớn nhất:", -heapq.heappop(h))
print("lấy lớn nhì :", -heapq.heappop(h))

heap (số âm): [-10, -9, -4, -3, -8]
lấy lớn nhất: 10
lấy lớn nhì : 9


## 3. Cách trực giác (baseline) — vét cạn mọi tập con

"Thử mọi cách chọn": với `k = 1, 2, ..., n`, thử **mọi** tập `k` phần tử (dùng `itertools.combinations`), giữ tập thỏa điều kiện có tổng lớn nhất. `k` đầu tiên tìm được là nhỏ nhất.

`combinations(nums, k)` sinh mọi bộ `k` phần tử (theo vị trí) không lặp.

In [6]:
from itertools import combinations

print(list(combinations([10, 9, 8], 2)))

[(10, 9), (10, 8), (9, 8)]


In [7]:
def min_subsequence_bruteforce(nums: list[int]) -> list[int]:
    total = sum(nums)
    for k in range(1, len(nums) + 1):
        best = None
        for combo in combinations(nums, k):
            s = sum(combo)
            if 2 * s > total and (best is None or s > sum(best)):
                best = combo
        if best is not None:
            return sorted(best, reverse=True)
    return []

for nums, expected in examples:
    print(nums, "->", min_subsequence_bruteforce(nums), "| kỳ vọng", expected)

[4, 3, 10, 9, 8] -> [10, 9] | kỳ vọng [10, 9]
[4, 4, 7, 6, 7] -> [7, 7, 6] | kỳ vọng [7, 7, 6]


### 3.1. Nút thắt

Số tập con là `2^n`. Với `n = 500`, `2^500` là con số khổng lồ — không bao giờ chạy xong. Baseline chỉ dùng được cho `n` rất nhỏ (≤ 15), và ta sẽ dùng nó làm **trọng tài** (oracle) để kiểm tra lời giải nhanh.

In [8]:
from math import comb
for n in [5, 10, 20, 500]:
    total_subsets = sum(comb(n, k) for k in range(1, n + 1))
    print(f"n = {n:>3}: số tập con khác rỗng = {total_subsets:.3e}")

n =   5: số tập con khác rỗng = 3.100e+01
n =  10: số tập con khác rỗng = 1.023e+03
n =  20: số tập con khác rỗng = 1.049e+06
n = 500: số tập con khác rỗng = 3.273e+150


## 4. Insight — tham lam: luôn lấy số lớn nhất còn lại

**Câu hỏi 1:** Nếu đã quyết định chọn đúng `k` phần tử, chọn những phần tử nào để tổng **lớn nhất**? → Chính là `k` phần tử **lớn nhất**.

**Câu hỏi 2:** Với `k` cố định, nếu ngay cả `k` phần tử lớn nhất cũng không vượt nửa tổng, thì **không** tập `k` phần tử nào vượt được.

⇒ `k` nhỏ nhất = số phần tử lớn nhất đầu tiên (theo thứ tự giảm dần) mà tổng của chúng `> total / 2`. Và tập đó cũng là tập có tổng lớn nhất trong các tập `k` phần tử ⇒ thỏa cả tiêu chí hòa.

Ví dụ `[4,4,7,6,7]`, `total = 28`: sắp giảm `7,7,6,4,4`; tổng dồn `7, 14, 20` → `2·20 = 40 > 28` ⇒ lấy `[7,7,6]`.

In [9]:
nums = [4, 4, 7, 6, 7]
total = sum(nums)
running = 0
for k, x in enumerate(sorted(nums, reverse=True), start=1):
    running += x
    print(f"k={k}: lấy {x}, tổng dồn={running}, 2*tổng={2 * running} > {total}? {2 * running > total}")
    if 2 * running > total:
        break

k=1: lấy 7, tổng dồn=7, 2*tổng=14 > 28? False
k=2: lấy 7, tổng dồn=14, 2*tổng=28 > 28? False
k=3: lấy 6, tổng dồn=20, 2*tổng=40 > 28? True


`enumerate(..., start=1)` đánh số từ 1 thay vì 0 — tiện để `k` là "số phần tử đã lấy".

## 5. Lời giải chính — sắp xếp + tham lam, giảng từng đoạn

**Đoạn 1:** `total = sum(nums)`.

**Đoạn 2:** sắp giảm dần: `ordered = sorted(nums, reverse=True)`.

**Đoạn 3:** duyệt, mỗi lần thêm `x` vào `result` và `chosen += x`.

**Đoạn 4:** ngay khi `2 * chosen > total` thì dừng, trả `result` (đã giảm dần vì ta lấy theo thứ tự giảm).

In [10]:
def min_subsequence_sort(nums: list[int]) -> list[int]:
    total = sum(nums)                      # Đoạn 1
    ordered = sorted(nums, reverse=True)   # Đoạn 2
    result, chosen = [], 0
    for x in ordered:                      # Đoạn 3
        result.append(x)
        chosen += x
        if 2 * chosen > total:             # Đoạn 4
            break
    return result

for nums, expected in examples:
    print(nums, "->", min_subsequence_sort(nums), "| kỳ vọng", expected)

[4, 3, 10, 9, 8] -> [10, 9] | kỳ vọng [10, 9]
[4, 4, 7, 6, 7] -> [7, 7, 6] | kỳ vọng [7, 7, 6]


**Vòng lặp chắc chắn dừng?** Có: khi lấy hết mọi phần tử, `chosen = total > 0` nên `2·total > total` (vì `nums[i] >= 1`). Vậy trường hợp xấu nhất là lấy cả mảng (ví dụ `[5]`).

### 5.1. Bản dùng slicing: tìm `k` rồi cắt `ordered[:k]`

`ordered[:k]` lấy `k` phần tử đầu (từ chỉ số 0 tới `k − 1`).

In [11]:
def min_subsequence_slice(nums: list[int]) -> list[int]:
    ordered = sorted(nums, reverse=True)
    total, chosen = sum(nums), 0
    for k, x in enumerate(ordered, start=1):
        chosen += x
        if 2 * chosen > total:
            return ordered[:k]
    return ordered

print([min_subsequence_slice(n) for n, _ in examples])
print("slicing mẫu:", [10, 9, 8, 4, 3][:2])

[[10, 9], [7, 7, 6]]
slicing mẫu: [10, 9]


## 6. Cách 3 — mảng đếm (counting sort), vì `1 <= nums[i] <= 100`

Không cần sắp xếp so sánh `O(n log n)`: đếm số lần mỗi giá trị `v` xuất hiện vào `count[v]`, rồi duyệt `v` từ `100` xuống `1`, lấy từng bản sao cho tới khi vượt nửa tổng. Thời gian `O(n + 100)`.

In [12]:
def min_subsequence_counting(nums: list[int]) -> list[int]:
    count = [0] * 101
    for x in nums:
        count[x] += 1
    total, chosen, result = sum(nums), 0, []
    for v in range(100, 0, -1):
        while count[v] > 0:
            result.append(v)
            chosen += v
            count[v] -= 1
            if 2 * chosen > total:
                return result
    return result

for nums, expected in examples:
    print(nums, "->", min_subsequence_counting(nums), "| kỳ vọng", expected)

[4, 3, 10, 9, 8] -> [10, 9] | kỳ vọng [10, 9]
[4, 4, 7, 6, 7] -> [7, 7, 6] | kỳ vọng [7, 7, 6]


## 7. Cách 4 — heap: chỉ lấy ra đúng số phần tử cần

Dựng max-heap (bằng số âm) trong `O(n)`, rồi `heappop` từng phần tử lớn nhất cho tới khi đủ. Nếu đáp án có `k` phần tử, chi phí là `O(n + k log n)` — có lợi khi `k` nhỏ hơn nhiều so với `n`.

In [13]:
def min_subsequence_heap(nums: list[int]) -> list[int]:
    h = [-x for x in nums]
    heapq.heapify(h)
    total, chosen, result = sum(nums), 0, []
    while 2 * chosen <= total:
        x = -heapq.heappop(h)
        result.append(x)
        chosen += x
    return result

for nums, expected in examples:
    print(nums, "->", min_subsequence_heap(nums), "| kỳ vọng", expected)

[4, 3, 10, 9, 8] -> [10, 9] | kỳ vọng [10, 9]
[4, 4, 7, 6, 7] -> [7, 7, 6] | kỳ vọng [7, 7, 6]


Điều kiện vòng lặp `while 2 * chosen <= total` là **phủ định** của điều kiện dừng `2 * chosen > total` — viết kiểu này thì không cần `break`.

## 8. TRACE từng bước

Trace cách sắp xếp với `nums = [4, 3, 10, 9, 8]` (`total = 34`):

| bước | x | chosen sau | 2·chosen | > 34? | result |
|---:|---:|---:|---:|:---:|---|
| 1 | 10 | 10 | 20 | ✘ | `[10]` |
| 2 | 9 | 19 | 38 | ✔ dừng | `[10, 9]` |

Với `nums = [4, 4, 7, 6, 7]` (`total = 28`):

| bước | x | chosen sau | 2·chosen | > 28? | result |
|---:|---:|---:|---:|:---:|---|
| 1 | 7 | 7 | 14 | ✘ | `[7]` |
| 2 | 7 | 14 | 28 | ✘ (**bằng**, không lớn hơn hẳn) | `[7, 7]` |
| 3 | 6 | 20 | 40 | ✔ dừng | `[7, 7, 6]` |

Bước 2 là nhánh dễ sai nhất: `14 = 14` **không** thỏa "strictly greater".

In [14]:
def trace_min_subsequence(nums: list[int]) -> list[int]:
    total = sum(nums)
    ordered = sorted(nums, reverse=True)
    print(f"nums={nums}, total={total}, ordered={ordered}")
    result, chosen = [], 0
    for step, x in enumerate(ordered, start=1):
        result.append(x)
        chosen += x
        ok = 2 * chosen > total
        print(f"  bước {step}: lấy {x:>3} -> chosen={chosen:>3}, rest={total - chosen:>3}, "
              f"chosen > rest? {ok} | result={result}")
        if ok:
            break
    return result

trace_min_subsequence([4, 3, 10, 9, 8])
print()
trace_min_subsequence([4, 4, 7, 6, 7])

nums=[4, 3, 10, 9, 8], total=34, ordered=[10, 9, 8, 4, 3]
  bước 1: lấy  10 -> chosen= 10, rest= 24, chosen > rest? False | result=[10]
  bước 2: lấy   9 -> chosen= 19, rest= 15, chosen > rest? True | result=[10, 9]

nums=[4, 4, 7, 6, 7], total=28, ordered=[7, 7, 6, 4, 4]
  bước 1: lấy   7 -> chosen=  7, rest= 21, chosen > rest? False | result=[7]
  bước 2: lấy   7 -> chosen= 14, rest= 14, chosen > rest? False | result=[7, 7]
  bước 3: lấy   6 -> chosen= 20, rest=  8, chosen > rest? True | result=[7, 7, 6]


[7, 7, 6]

Trace trường hợp **phải lấy gần hết** — tất cả bằng nhau:

In [15]:
trace_min_subsequence([5, 5, 5, 5])

nums=[5, 5, 5, 5], total=20, ordered=[5, 5, 5, 5]
  bước 1: lấy   5 -> chosen=  5, rest= 15, chosen > rest? False | result=[5]
  bước 2: lấy   5 -> chosen= 10, rest= 10, chosen > rest? False | result=[5, 5]
  bước 3: lấy   5 -> chosen= 15, rest=  5, chosen > rest? True | result=[5, 5, 5]


[5, 5, 5]

4 phần tử bằng 5: lấy 2 thì `10 = 10` chưa đủ, phải lấy 3 (`15 > 5`).

## 9. Code nộp LeetCode

In [16]:
class Solution:
    def minSubsequence(self, nums: list[int]) -> list[int]:
        nums = sorted(nums, reverse=True)
        total, chosen = sum(nums), 0
        for k, x in enumerate(nums, start=1):
            chosen += x
            if 2 * chosen > total:
                return nums[:k]
        return nums

print([Solution().minSubsequence(n) for n, _ in examples])

[[10, 9], [7, 7, 6]]


### Bản counting sort (nhanh nhất theo lý thuyết)

In [17]:
class SolutionCounting:
    def minSubsequence(self, nums: list[int]) -> list[int]:
        count = [0] * 101
        for x in nums:
            count[x] += 1
        half, chosen, res = sum(nums), 0, []
        for v in range(100, 0, -1):
            for _ in range(count[v]):
                res.append(v)
                chosen += v
                if 2 * chosen > half:
                    return res
        return res

print([SolutionCounting().minSubsequence(n) for n, _ in examples])

[[10, 9], [7, 7, 6]]


`for _ in range(count[v])` lặp `count[v]` lần; dấu `_` là quy ước "biến không dùng tới".

### Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi tập con theo kích thước | `O(2^n · n)` | `O(n)` | chỉ dùng làm oracle |
| 2. Sắp xếp + tham lam | lấy lớn nhất tới khi > nửa | `O(n log n)` | `O(n)` | chuẩn, dễ nhớ |
| 3. Counting sort | duyệt giá trị 100 → 1 | `O(n + V)` | `O(V + n)` | `V = 100` |
| 4. Heap | pop lớn nhất `k` lần | `O(n + k log n)` | `O(n)` | lợi khi `k` nhỏ |

## 10. Vì sao tham lam đúng?

Gọi `a_1 >= a_2 >= ... >= a_n` là mảng sắp giảm, `P_k = a_1 + ... + a_k`, `T = P_n`.

**Bổ đề:** trong mọi tập con có đúng `k` phần tử, tổng lớn nhất là `P_k`. (Thay bất kỳ phần tử nào ngoài top-`k` bằng một phần tử trong top-`k` chưa chọn thì tổng không giảm — lập luận **đổi chỗ** / exchange argument.)

**Kích thước nhỏ nhất:** gọi `k*` là chỉ số nhỏ nhất có `2·P_k > T`. Với mọi `k < k*`: mọi tập `k` phần tử có tổng `<= P_k`, mà `2·P_k <= T` ⇒ không thỏa. Vậy không tồn tại đáp án ít hơn `k*` phần tử; và top-`k*` thỏa ⇒ kích thước tối thiểu là `k*`.

**Tổng lớn nhất trong các tập `k*` phần tử:** theo bổ đề, là top-`k*`. ∎

**Bất biến vòng lặp:** sau khi xử lý `k` phần tử, `result = [a_1..a_k]` và `chosen = P_k`; vòng lặp dừng tại `k` đầu tiên có `2·P_k > T` ⇒ chính là `k*`.

## 11. Time / Space

Gọi `n = len(nums)`, `V = 100` (giá trị lớn nhất).

- **Sắp xếp:** `O(n log n)` cho `sorted`, `O(n)` cho vòng lặp ⇒ `O(n log n)`. Bộ nhớ `O(n)` (list đã sắp; Timsort cần thêm tối đa `O(n)`).
- **Counting:** `O(n + V)` thời gian; `O(V)` cho mảng đếm + `O(k)` cho kết quả.
- **Heap:** `heapify` `O(n)` + `k` lần `heappop` `O(log n)`.
- **Vét cạn:** `O(2^n · n)`.

In [18]:
import random, time
random.seed(1)
big = [random.randint(1, 100) for _ in range(500)]
for f in [min_subsequence_sort, min_subsequence_counting, min_subsequence_heap]:
    t0 = time.perf_counter()
    for _ in range(500):
        f(big)
    print(f"{f.__name__:<26} {1000 * (time.perf_counter() - t0):7.2f} ms / 500 lần, k = {len(f(big))}")

min_subsequence_sort         24.88 ms / 500 lần, k = 146
min_subsequence_counting     24.72 ms / 500 lần, k = 146


min_subsequence_heap         42.57 ms / 500 lần, k = 146


## 12. Bộ test

Mẫu đề, 1 phần tử, tất cả bằng nhau (chẵn/lẻ số lượng), một phần tử áp đảo, trường hợp "bằng đúng nửa", và **đối chiếu với vét cạn** trên mảng ngẫu nhiên nhỏ.

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS -> {actual}")

sol = Solution()
check("ví dụ 1", sol.minSubsequence([4, 3, 10, 9, 8]), [10, 9])
check("ví dụ 2", sol.minSubsequence([4, 4, 7, 6, 7]), [7, 7, 6])
check("một phần tử", sol.minSubsequence([6]), [6])
check("4 số bằng nhau", sol.minSubsequence([5, 5, 5, 5]), [5, 5, 5])
check("3 số bằng nhau", sol.minSubsequence([2, 2, 2]), [2, 2])
check("một số áp đảo", sol.minSubsequence([1, 1, 100, 1]), [100])
check("bằng đúng nửa -> phải thêm", sol.minSubsequence([6, 3, 3]), [6, 3])
check("hai số", sol.minSubsequence([1, 2]), [2])

ví dụ 1                      PASS -> [10, 9]
ví dụ 2                      PASS -> [7, 7, 6]
một phần tử                  PASS -> [6]
4 số bằng nhau               PASS -> [5, 5, 5]
3 số bằng nhau               PASS -> [2, 2]
một số áp đảo                PASS -> [100]
bằng đúng nửa -> phải thêm   PASS -> [6, 3]
hai số                       PASS -> [2]


In [20]:
methods = [min_subsequence_sort, min_subsequence_slice, min_subsequence_counting, min_subsequence_heap,
           Solution().minSubsequence, SolutionCounting().minSubsequence]
random.seed(1403)
for _ in range(600):
    nums = [random.randint(1, 12) for _ in range(random.randint(1, 9))]
    expected = min_subsequence_bruteforce(nums)
    for f in methods:
        assert f(list(nums)) == expected, (f, nums)
print("Mọi cách khớp vét cạn trên 600 mảng ngẫu nhiên")

Mọi cách khớp vét cạn trên 600 mảng ngẫu nhiên


## 13. Lỗi người mới hay gặp

### Lỗi 1: Dùng `>=` thay vì `>` (quên "strictly greater")

In [21]:
def wrong_ge(nums):
    nums = sorted(nums, reverse=True)
    total, chosen = sum(nums), 0
    for k, x in enumerate(nums, start=1):
        chosen += x
        if 2 * chosen >= total:          # SAI: bằng nhau chưa đủ
            return nums[:k]
    return nums

print("sai:", wrong_ge([4, 4, 7, 6, 7]), "| đúng:", min_subsequence_sort([4, 4, 7, 6, 7]))

sai: [7, 7] | đúng: [7, 7, 6]


### Lỗi 2: Sắp tăng dần rồi lấy từ đầu (lấy số nhỏ trước)

In [22]:
def wrong_ascending(nums):
    nums = sorted(nums)                  # SAI: tăng dần
    total, chosen, res = sum(nums), 0, []
    for x in nums:
        res.append(x)
        chosen += x
        if 2 * chosen > total:
            break
    return res

print("sai:", wrong_ascending([4, 3, 10, 9, 8]), "| đúng:", min_subsequence_sort([4, 3, 10, 9, 8]))

sai: [3, 4, 8, 9] | đúng: [10, 9]


### Lỗi 3: Dùng `nums.sort()` rồi gán `nums = nums.sort()`

`.sort()` trả `None` ⇒ biến `nums` thành `None` và lỗi `TypeError` ở dòng sau.

In [23]:
nums = [3, 1, 2]
nums = nums.sort(reverse=True)
print("nums =", nums)
try:
    sum(nums)
except TypeError as e:
    print("TypeError:", e)

nums = None
TypeError: 'NoneType' object is not iterable


### Lỗi 4: Tính `rest` bằng cách xóa phần tử trong lúc duyệt

Xóa phần tử khỏi list đang duyệt (`for x in nums: nums.remove(x)`) làm vòng lặp **bỏ qua** phần tử. Hãy dùng công thức `rest = total − chosen` thay vì sửa list.

In [24]:
nums = [10, 9, 8, 4, 3]
seen = []
for x in nums:
    seen.append(x)
    nums.remove(x)        # SAI: sửa list khi đang duyệt
print("đã duyệt:", seen, "| list còn lại:", nums)

đã duyệt: [10, 8, 3] | list còn lại: [9, 4]


## 14. Mẹo nhận dạng pattern

- "Chọn **ít phần tử nhất** để tổng vượt ngưỡng" ⇒ tham lam: lấy phần tử **lớn nhất** trước (sắp giảm / heap).
- "Tổng phần chọn > tổng phần còn lại" ⇒ đổi thành `2 · chosen > total`.
- Giá trị nhỏ, biết trước (≤ 100, ≤ 1000) ⇒ có thể thay sắp xếp bằng **mảng đếm**.
- Khi gặp "subsequence" nhưng đáp án chỉ phụ thuộc tổng/kích thước ⇒ thứ tự không quan trọng, được phép sắp xếp.
- Chứng minh tham lam bằng **exchange argument**: đổi một lựa chọn "kém" sang lựa chọn tham lam không làm tệ hơn.

## 15. Biến thể

1. Trả về **số phần tử tối thiểu** `k` thay vì dãy.
2. Ngưỡng tổng quát: tổng chọn `>= target` (LeetCode 1338-style "giảm kích thước mảng một nửa" dùng cùng ý tưởng với tần suất).
3. Tổng phần chọn phải **lớn hơn gấp đôi** phần còn lại: `chosen > 2·(total − chosen)` ⇔ `3·chosen > 2·total`.
4. LeetCode 1338 *Reduce Array Size to The Half*: chọn ít **giá trị** nhất để xóa ít nhất một nửa mảng — tham lam theo **tần suất** giảm dần.

In [25]:
from collections import Counter

def min_count_to_exceed(nums: list[int], target: int) -> int:
    chosen = 0
    for k, x in enumerate(sorted(nums, reverse=True), start=1):
        chosen += x
        if chosen >= target:
            return k
    return -1

def min_set_size(arr: list[int]) -> int:          # LeetCode 1338
    removed, half = 0, len(arr) // 2
    for k, f in enumerate(sorted(Counter(arr).values(), reverse=True), start=1):
        removed += f
        if removed >= half:
            return k
    return len(arr)

print(min_count_to_exceed([4, 3, 10, 9, 8], 25))
print(min_set_size([3, 3, 3, 3, 5, 5, 5, 2, 2, 7]))

3
2


## 16. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** `nums = [1, 7, 3, 7, 2, 5]`. Tổng = ? Sắp giảm, lập bảng tổng dồn, cho đáp án.

**Bài 2 (đoán output):** `sorted([3, 1, 2], reverse=True)[:2]`, `[5, 4][:5]`, `2 * 14 > 28`, `[1, 3, 2].sort()`.

**Bài 3 (điền chỗ trống):** `exercise_min_subsequence(nums)` dùng `sorted` + vòng lặp tham lam.

**Bài 4 (tự viết):** `exercise_min_subsequence_counting(nums)` bằng mảng đếm 101 ô.

**Bài 5 (biến thể):** `exercise_min_size_double(nums)` — trả **số phần tử ít nhất** sao cho tổng phần chọn **lớn hơn gấp đôi** tổng phần còn lại.

**Bài 6 (mini challenge):** `exercise_min_subsequence_keep_order(nums)` — trả cùng tập phần tử như đáp án, nhưng theo **thứ tự xuất hiện gốc** trong `nums` (dãy con đúng nghĩa). Đề bảo đảm hoặc giả sử các giá trị đôi một khác nhau.

In [26]:
def exercise_min_subsequence(nums: list[int]) -> list[int]:
    # TODO 1: total = sum(nums)
    # TODO 2: ordered = sorted(..., reverse=True)
    # TODO 3: cộng dồn, dừng khi 2 * chosen > total
    pass

def exercise_min_subsequence_counting(nums: list[int]) -> list[int]:
    # TODO
    pass

# assert exercise_min_subsequence([4, 3, 10, 9, 8]) == [10, 9]
# assert exercise_min_subsequence_counting([4, 4, 7, 6, 7]) == [7, 7, 6]

In [27]:
def exercise_min_size_double(nums: list[int]) -> int:
    # TODO: điều kiện 3 * chosen > 2 * total
    pass

def exercise_min_subsequence_keep_order(nums: list[int]) -> list[int]:
    # TODO
    pass

# assert exercise_min_size_double([4, 3, 10, 9, 8]) == 3
# assert exercise_min_subsequence_keep_order([4, 3, 10, 9, 8]) == [10, 9]
# assert exercise_min_subsequence_keep_order([9, 1, 10, 2]) == [9, 10]

## 17. Gợi ý 3 mức

**Bài 1** — (1) Tổng = 25. (2) Sắp giảm `7, 7, 5, 3, 2, 1`. (3) Tổng dồn `7, 14` → `28 > 25`?

**Bài 3** — (1) Ba đoạn: tổng, sắp, cộng dồn. (2) `result.append(x)`; `chosen += x`. (3) `if 2 * chosen > total: break`.

**Bài 4** — (1) `count = [0] * 101`. (2) `for v in range(100, 0, -1)`. (3) Vòng trong lặp `count[v]` lần, kiểm tra điều kiện sau mỗi lần lấy.

**Bài 5** — (1) `chosen > 2 * (total - chosen)`. (2) ⇔ `3 * chosen > 2 * total`. (3) Trả `k` thay vì list.

**Bài 6** — (1) Tính đáp án như bài chính để biết **tập** giá trị. (2) Duyệt `nums` theo thứ tự gốc. (3) Giữ phần tử nào nằm trong tập (dùng `Counter` để xử lý cả giá trị lặp nếu muốn tổng quát).

## 18. Đáp án tham khảo

**Bài 1:** tổng 25; tổng dồn `7 → 14`, `2·14 = 28 > 25` ⇒ **`[7, 7]`**.

**Bài 2:** `[3, 2]`, `[5, 4]` (cắt quá độ dài không lỗi), `False`, `None`.

In [28]:
def answer_min_subsequence(nums):
    total = sum(nums)
    ordered = sorted(nums, reverse=True)
    result, chosen = [], 0
    for x in ordered:
        result.append(x)
        chosen += x
        if 2 * chosen > total:
            break
    return result

def answer_min_subsequence_counting(nums):
    count = [0] * 101
    for x in nums:
        count[x] += 1
    total, chosen, res = sum(nums), 0, []
    for v in range(100, 0, -1):
        for _ in range(count[v]):
            res.append(v)
            chosen += v
            if 2 * chosen > total:
                return res
    return res

def answer_min_size_double(nums):
    total, chosen = sum(nums), 0
    for k, x in enumerate(sorted(nums, reverse=True), start=1):
        chosen += x
        if 3 * chosen > 2 * total:
            return k
    return len(nums)

def answer_min_subsequence_keep_order(nums):
    need = Counter(answer_min_subsequence(nums))
    res = []
    for x in nums:
        if need[x] > 0:
            res.append(x)
            need[x] -= 1
    return res

assert answer_min_subsequence([1, 7, 3, 7, 2, 5]) == [7, 7]
assert sorted([3, 1, 2], reverse=True)[:2] == [3, 2] and [5, 4][:5] == [5, 4]
assert (2 * 14 > 28) is False and [1, 3, 2].sort() is None
assert answer_min_subsequence([4, 3, 10, 9, 8]) == [10, 9]
assert answer_min_subsequence_counting([4, 4, 7, 6, 7]) == [7, 7, 6]
assert answer_min_size_double([4, 3, 10, 9, 8]) == 3
assert answer_min_subsequence_keep_order([4, 3, 10, 9, 8]) == [10, 9]
assert answer_min_subsequence_keep_order([9, 1, 10, 2]) == [9, 10]
random.seed(99)
for _ in range(300):
    nums = [random.randint(1, 100) for _ in range(random.randint(1, 50))]
    assert answer_min_subsequence(nums) == answer_min_subsequence_counting(nums) == min_subsequence_heap(nums)
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


Kiểm tra lại bài 5 với `[4,3,10,9,8]` (`total = 34`): lấy `10, 9` → `chosen = 19`, `rest = 15`, `19 > 30`? Không. Thêm `8` → `27 > 2·7 = 14` ✔ ⇒ `3` phần tử.

## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def minSubsequence(self, nums: list[int]) -> list[int]:
        nums = sorted(nums, reverse=True)          # 1. lớn nhất trước
        total, chosen = sum(nums), 0
        for k, x in enumerate(nums, start=1):
            chosen += x                            # 2. cộng dồn
            if 2 * chosen > total:                 # 3. "lớn hơn hẳn" phần còn lại
                return nums[:k]
        return nums
```

**Khung tham lam "ít phần tử nhất để vượt ngưỡng":** sắp giảm → cộng dồn → dừng khi vượt ngưỡng → chứng minh bằng exchange argument.

**Checklist:**
- [ ] Dùng `>` (strictly greater), không phải `>=`.
- [ ] Sắp **giảm dần**; output cũng phải giảm dần.
- [ ] Không gán `nums = nums.sort()`.
- [ ] Dùng `2 * chosen > total` để khỏi tính `rest` và tránh số thực.